# JARVIS — Ollama on Cloud GPU (free T4)

Runs Ollama on Colab's free T4 GPU and exposes it over a public HTTPS tunnel so your local JARVIS (opencode) can use it.

**Before running:**
1. Menu `Runtime → Change runtime type → T4 GPU → Save`
2. Menu `Runtime → Run all`
3. Copy the `https://....trycloudflare.com` URL printed by the last cell and share it with your local setup.

In [ ]:
# 1. Install Ollama into the Colab environment
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2. Install cloudflared (the free tunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared && cloudflared --version

In [ ]:
# 3. Start the Ollama server in the background
import os, subprocess, time, re, threading
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'  # keep model resident in memory -> faster after first call
subprocess.Popen(['nohup', 'ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for the API to come up
for _ in range(60):
    try:
        import urllib.request
        urllib.request.urlopen('http://localhost:11434/api/version', timeout=2)
        print('Ollama server is up.')
        break
    except Exception:
        time.sleep(2)


In [ ]:
# 4. Pull the model (qwen3:8b — same as your local one). Takes a few minutes on first run.
MODEL = 'qwen3:8b'
import subprocess
p = subprocess.Popen(['ollama', 'pull', MODEL], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in p.stdout:
    print(line, end='')
p.wait()
print('\npull done')

In [ ]:
# 5. Open a Cloudflare quick tunnel to localhost:11434
import subprocess, re, threading
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate', '--no-tls-verify'],
    stdout=subprocess.DEVNULL, stderr=subprocess.PIPE
)
public_url = None
for _ in range(120):
    line = cf.stderr.readline().decode(errors='ignore')
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    raise RuntimeError('Could not establish Cloudflare tunnel.')

BASE = public_url.rstrip('/')
print('======================================================')
print('JARVIS CLOUD OLLAMA READY')
print('Base URL :', BASE)
print('Model    :', MODEL)
print('OpenAI  /v1 endpoint:', BASE + '/v1')
print('\nShare this Base URL with your local setup:')
print(BASE)
print('======================================================')

# Keep the cell alive (stop the cell with ■ to shut down cleanly)
while True:
    time.sleep(3600)